# Notebook 1 — Bayesian Classification (Mixed Features, Shelter Outcomes)

**Filename required:** `i24-XXXX_First-Last_1_BayesianClassification.ipynb`  
**Allowed libraries:** `numpy`, `math` only (no pandas, sklearn, nltk, etc.)  
**Dataset:** Teacher-provided Shelter Animal Outcomes subset CSV (columns mapped/obfuscated).  

## Objective
Implement a **mixed Bayesian classifier from scratch** (categorical Naïve Bayes + Gaussian Naïve Bayes) using only `numpy` and `math`. No external ML or data libraries are allowed. All helper utilities and evaluation must be written from scratch.

## Classes (collapsed to 4)
`{Adoption, Transfer, Return_to_Owner, Other}`

## Strict Preprocessing Rules (MUST follow)
- **Derive features:**
  - `age_days` from `Age` string (rules: years=365, months=30, weeks=7, days=1; unknown/empty → 0)
  - `time_in_shelter_days = ceil(OutDt − InDt)` (if invalid/missing → 0)
- **Encodings:**
  - `AnimalType`: Dog→1, Cat→0, else 0
  - `SexUponIntake` (SxIn):
    - `Neutered Male` or `Spayed Female` → 2
    - `Intact Male` or `Intact Female` → 1
    - unknown/empty/other → 0
  - `Color` (Clr): **first token** only (split on space or '/'), uppercase; empty → `UNKNOWN`
  - `Breed` (Br): keep **top-20** by **TRAIN** frequency; others → `Other`; empty → `Unknown` (then rule)
- **Scaling:** z-score numerical features (`age_days`, `time_in_shelter_days`) with μ,σ² **from TRAIN only**; clamp σ² ≥ 1e−6.
- **Split:** stratified **80/20** (numpy-only). Seed = `(ROLL % 97) + 7`.
- **Smoothing:** `K = 0.5 + ((sum of digits of ROLL) % 5)/10`  → value in `[0.5, 0.9]`.

## Mandatory Logs (PRINT)
1. Class counts after collapse (TRAIN)  
2. Top-20 breeds + counts (TRAIN)  
3. Log-priors per class  
4. For test row index `r = (ROLL % 97)`:  
   - Each categorical feature’s value & per-class **log P(value|class)**  
   - Gaussian stats per class (μ, σ²) for numeric features  
   - Final class-wise log-posterior & predicted class

## Evaluation (same across all notebooks)
- Accuracy, per-class Precision/Recall/F1, Macro-F1, Confusion Matrix (C×C) — **numpy-only**.

## Submission
- This notebook with all required function signatures filled.
- A **Report PDF** named `i24-XXXX_FirstName-LastName_Report.pdf` with fixed structure:
  1) Overview, 2) Preprocessing, 3) Model, 4) Results, 5) Analysis, 6) Limitations & Future Work, 7) Worked Example.

---
**Do not edit the function signatures below.** Only `numpy` and `math` imports are permitted.


In [ ]:
import numpy as np
import math

# === Student roll number ===
ROLL = 230000  # << EDIT THIS: your roll number as integer
np.random.seed((ROLL % 97) + 7)

def smoothing_k(roll: int) -> float:
    """
    Expectation:
    - Input: integer roll number
    - Output: Laplace smoothing constant K in [0.5, 0.9]
    - Rule: 0.5 + ((sum of digits of roll) % 5)/10.0
    """
    s = sum(int(d) for d in str(roll))
    return 0.5 + (s % 5)/10.0

K = smoothing_k(ROLL)
K

## Required: Parsing & Derived Features

In [ ]:
def parse_age_to_days(age_str: str) -> int:
    """
    Expectation:
    - Input: string like '2 years', '3 months', '4 weeks', '10 days'.
    - Conversion rules: years=365, months=30, weeks=7, days=1.
    - If empty/unknown/invalid: return 0.
    - Output: integer number of days (>=0).
    """
    pass

def days_between_iso(intake_iso: str, outcome_iso: str) -> int:
    """
    Expectation:
    - Input: ISO-like strings 'YYYY-MM-DD HH:MM:SS' for intake & outcome.
    - Output: ceil(#days) between outcome and intake.
    - If outcome is missing/invalid or earlier than intake, return 0.
    - Pure numpy/math/string ops; no datetime library usage required.
    """
    pass

def first_token(s: str) -> str:
    """
    Expectation:
    - Input: a color string possibly containing spaces or '/'.
    - Return: the first token (split by space or '/'), in UPPERCASE.
    - Empty/None -> 'UNKNOWN'.
    """
    pass

def top_k(values: np.ndarray, k: int) -> np.ndarray:
    """
    Expectation:
    - Input: 1D array of strings (categories), integer k.
    - Output: array (len<=k) of top-k categories by frequency.
    - Tie-break: lexicographical order.
    - No pandas; use numpy & pure Python only.
    """
    pass


## Required: Encoders & Scaling

In [ ]:
def map_animal_type(x: str) -> int:
    """
    Expectation:
    - Input: animal type string; treat 'Dog'->1, 'Cat'->0, else 0.
    - Output: integer 0 or 1.
    """
    pass

def map_sex_upon_intake(x: str) -> int:
    """
    Expectation:
    - Input: SexUponIntake string.
    - Map: 'Neutered Male' or 'Spayed Female' -> 2
            'Intact Male' or 'Intact Female'  -> 1
            unknown/empty/other               -> 0
    - Output: integer in {0,1,2}.
    """
    pass

def encode_breed(breed: str, keep_set: set) -> str:
    """
    Expectation:
    - Input: breed string and a set of kept breeds (top-20 by TRAIN frequency).
    - Behavior: return breed if in keep_set; else 'Other'.
    - If empty/None: treat as 'Unknown' then apply rule.
    - Output: string category.
    """
    pass

def zscore_fit(x: np.ndarray) -> tuple[float, float]:
    """
    Expectation:
    - Input: 1D numeric array (TRAIN only).
    - Output: (mean, var) with var >= 1e-6 (clamp if necessary).
    """
    pass

def zscore_apply(x: np.ndarray, mean: float, var: float) -> np.ndarray:
    """
    Expectation:
    - Input: 1D numeric array and (mean, var) from TRAIN.
    - Output: z-scored array: (x - mean)/sqrt(var).
    """
    pass


## Required: Splitting (Stratified 80/20)

In [ ]:
def stratified_split_np(y: np.ndarray, test_ratio: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Expectation:
    - Input: labels y (shape (N,)), test_ratio (e.g., 0.2), seed (int).
    - Output: (train_indices, test_indices) as 1D integer arrays.
    - Must be stratified by class label proportions using numpy-only ops.
    - Deterministic under the given seed.
    """
    pass


## Required: Naïve Bayes Core (Mixed)

In [ ]:
def logsumexp(v: np.ndarray) -> float:
    """
    Expectation:
    - Input: 1D array of log-values.
    - Output: log(sum(exp(v))) computed stably.
    - Numerically stable implementation required.
    """
    pass

def fit_nb_categorical(X_cat: np.ndarray, y: np.ndarray, K: float) -> dict:
    """
    Expectation:
    - Inputs:
        X_cat: shape (N, F_cat), integer-encoded categorical features.
        y:     shape (N,), class labels as ints [0..C-1].
        K:     Laplace smoothing constant.
    - Process:
        1) For each feature f, for each (value v, class c), count occurrences.
        2) Apply Laplace smoothing with K.
        3) Compute log P(value=v | class=c, feature=f).
    - Output:
        A dict containing per-feature tables with log-probabilities and the
        necessary metadata to map values at inference time.
    """
    pass

def fit_nb_gaussian(X_num: np.ndarray, y: np.ndarray) -> dict:
    """
    Expectation:
    - Inputs:
        X_num: shape (N, F_num), z-scored numeric features.
        y:     shape (N,), class labels as ints [0..C-1].
    - Process:
        For each class c and numeric feature j, compute mean and variance.
        Ensure variance >= 1e-6.
    - Output:
        A dict mapping class index to arrays of means and variances per numeric feature.
    """
    pass

def fit_prior(y: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Input: labels y (shape (N,)).
    - Output: log-prior probabilities for each class as a 1D array (length C).
    - Empirical priors based on TRAIN frequency.
    """
    pass

def predict_log_proba(model: dict, X_cat: np.ndarray, X_num: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Inputs:
        model: a dict with keys {'cat', 'gauss', 'log_prior', 'cat_value_maps', 'gauss_stats'}
        X_cat: (N, F_cat) integer-encoded categorical features.
        X_num: (N, F_num) z-scored numeric features.
    - Process:
        For each sample, compute log-posterior per class:
          log P(c) + sum_f log P_cat(x_f|c) + sum_j log N(x_j|mu_{c,j}, var_{c,j}).
    - Output:
        (N, C) matrix of log-posteriors.
    """
    pass

def to_proba(logps: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Input: (N, C) matrix of log-posteriors.
    - Output: (N, C) probabilities normalized to sum to 1 along C using logsumexp.
    """
    pass

def predict(model: dict, X_cat: np.ndarray, X_num: np.ndarray) -> np.ndarray:
    """
    Expectation:
    - Output: 1D array of predicted class indices via argmax over log-posteriors.
    """
    pass


## Required: Evaluation Metrics (numpy-only)

In [ ]:
def confusion_matrix_np(y_true: np.ndarray, y_pred: np.ndarray, C: int) -> np.ndarray:
    """
    Expectation:
    - Inputs: y_true, y_pred (shape (N,)), number of classes C.
    - Output: (C, C) integer confusion matrix where rows=true, cols=pred.
    - Must be implemented with numpy operations only.
    """
    pass

def precision_recall_f1_per_class(cm: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Expectation:
    - Input: confusion matrix cm of shape (C, C).
    - Output: (precisions, recalls, f1s) each shape (C,).
    - Zero-division should yield 0 for the corresponding metric.
    """
    pass

def macro_f1(prec: np.ndarray, rec: np.ndarray, f1: np.ndarray) -> float:
    """
    Expectation:
    - Input: per-class precision, recall, f1 arrays (shape (C,)).
    - Output: scalar macro-F1 = mean(f1).
    """
    pass


## Pipeline Skeleton (fill with your code)
- Load CSV (teacher-provided) using Python file I/O or `np.genfromtxt` (no pandas).
- Apply strict preprocessing rules.
- Build categorical value maps (e.g., Breed top-20 from TRAIN only).
- Build `X_cat`, `X_num`, `y` for TRAIN/TEST.
- Fit priors, categorical NB, gaussian NB.
- Predict on TEST; print mandatory logs and metrics.

In [ ]:
# === YOUR DATA LOADING & PREPROCESSING CODE HERE ===
# Reminder: Obey all strict rules listed above.

# Example (pseudo-structure):
# raw = ...  # read CSV lines
# parse columns: A_Type, SxIn, Br, Clr, Age, InDt, Out, OutDt
# derive age_days, time_in_shelter_days
# collapse classes to 4
# split stratified 80/20 -> train_idx, test_idx
# compute top-20 breeds from TRAIN only
# encode categorical (int codes per feature), z-score numeric
# build X_cat_train, X_num_train, y_train, and test counterparts
pass


## Fit Model & Mandatory Logs (fill with your code)

In [ ]:
# === FIT PRIORS ===
# log_prior = fit_prior(y_train)

# === FIT CATEGORICAL NB ===
# cat_model = fit_nb_categorical(X_cat_train, y_train, K)

# === FIT GAUSSIAN NB ===
# gauss_model = fit_nb_gaussian(X_num_train, y_train)

# model = {
#   'log_prior': log_prior,
#   'cat': cat_model,
#   'gauss': gauss_model,
#   'cat_value_maps': None,   # store any mapping metadata you need
#   'gauss_stats': None       # optional, if you want to store (mu/var) in a flat structure
# }

# === MANDATORY LOGS ===
# 1) Class counts (TRAIN)
# 2) Top-20 breeds + counts (TRAIN)
# 3) Log-priors per class
# 4) For r = (ROLL % 97): categorical per-class log-liks, gaussian μ/σ², final log-posteriors, predicted class
pass


## Inference & Evaluation (fill with your code)

In [ ]:
# === PREDICT ===
# logps_test = predict_log_proba(model, X_cat_test, X_num_test)
# y_pred = np.argmax(logps_test, axis=1)

# === METRICS ===
# C = number_of_classes
# cm = confusion_matrix_np(y_test, y_pred, C)
# prec, rec, f1 = precision_recall_f1_per_class(cm)
# acc = (y_test == y_pred).mean()
# macro = macro_f1(prec, rec, f1)

# print('Accuracy:', float(acc))
# print('Precision per class:', prec)
# print('Recall per class:', rec)
# print('F1 per class:', f1)
# print('Macro-F1:', float(macro))
pass


## Submission Checks (self-verify)
- [ ] All required function signatures are implemented (no `pass`).
- [ ] Only `numpy` & `math` used (no pandas/sklearn/etc.).
- [ ] Split is stratified 80/20 with the specified seed.
- [ ] Mandatory logs are printed.
- [ ] Metrics computed via numpy-only functions.
- [ ] Report PDF prepared with fixed structure.
